# Dynamic Question Router - Google Colab Demo

This notebook demonstrates the Dynamic Question Router system in Google Colab.

## Features
- Question classification (domain and difficulty)
- Intelligent routing to specialized chatbots
- Feedback collection system
- Fallback mechanism for API failures

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q fastapi uvicorn[standard] pydantic python-multipart
!pip install -q transformers torch accelerate
!pip install -q openai anthropic requests pandas aiofiles python-dotenv
!pip install -q pyngrok

print("✅ All dependencies installed!")

## Step 2: Clone Repository (if needed)

In [ ]:
# Clone the repository
!git clone https://github.com/VARUNVARSHAN-RN/DYNAMIC-QUESTION-ROUTER.git
%cd DYNAMIC-QUESTION-ROUTER

print("✅ Repository cloned!")

## Step 3: Configure API Keys

In [ ]:
import os

# Set your API keys here
# You can get these keys from:
# - OpenAI: https://platform.openai.com/api-keys
# - DeepSeek: https://platform.deepseek.com/
# - Anthropic: https://console.anthropic.com/
# - HuggingFace: https://huggingface.co/settings/tokens

os.environ['OPENAI_API_KEY'] = 'your_openai_key_here'
os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_key_here'
os.environ['ANTHROPIC_API_KEY'] = 'your_anthropic_key_here'
os.environ['HUGGINGFACE_API_KEY'] = 'your_huggingface_key_here'

print("✅ API keys configured!")
print("⚠️  Remember to replace 'your_*_key_here' with your actual API keys")

## Step 4: Start the Server

In [ ]:
from pyngrok import ngrok
import threading
import time

# Set up ngrok tunnel
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")
print(f"📚 API Docs: {public_url}/docs")

# Start FastAPI server in background thread
def run_server():
    import uvicorn
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(5)
print("\n✅ Server started!")

# Store the URL for later use
api_url = str(public_url)

## Step 5: Test the API

### Test 1: Health Check

In [ ]:
import requests
import json

response = requests.get(f"{api_url}/health")
print("Health Check:")
print(json.dumps(response.json(), indent=2))

### Test 2: Ask a Math Question

In [ ]:
response = requests.post(
    f"{api_url}/ask",
    json={"question": "What is the derivative of x^2 + 3x + 5?"}
)
print("Math Question:")
result = response.json()
print(json.dumps(result, indent=2))

# Save for feedback later
math_result = result

### Test 3: Ask a Coding Question

In [ ]:
response = requests.post(
    f"{api_url}/ask",
    json={"question": "How do I implement a binary search algorithm in Python?"}
)
print("Coding Question:")
result = response.json()
print(json.dumps(result, indent=2))

coding_result = result

### Test 4: Ask a Science Question

In [ ]:
response = requests.post(
    f"{api_url}/ask",
    json={"question": "Explain how photosynthesis works at the molecular level"}
)
print("Science Question:")
print(json.dumps(response.json(), indent=2))

### Test 5: Submit Positive Feedback

In [ ]:
feedback_response = requests.post(
    f"{api_url}/feedback",
    json={
        "question": math_result["question"],
        "predicted_domain": math_result["domain"],
        "predicted_difficulty": math_result["difficulty"],
        "chatbot_used": math_result["chatbot_used"],
        "response": math_result["answer"],
        "feedback_positive": True
    }
)
print("Feedback Response:")
print(json.dumps(feedback_response.json(), indent=2))

### Test 6: Submit Negative Feedback with Corrections

In [ ]:
feedback_response = requests.post(
    f"{api_url}/feedback",
    json={
        "question": coding_result["question"],
        "predicted_domain": coding_result["domain"],
        "predicted_difficulty": coding_result["difficulty"],
        "chatbot_used": coding_result["chatbot_used"],
        "response": coding_result["answer"],
        "feedback_positive": False,
        "corrected_domain": "Coding",
        "corrected_difficulty": "Hard"
    }
)
print("Feedback Response:")
print(json.dumps(feedback_response.json(), indent=2))

### Test 7: Get Statistics

In [ ]:
stats_response = requests.get(f"{api_url}/stats")
print("Statistics:")
print(json.dumps(stats_response.json(), indent=2))

## Custom Questions

Try your own questions below:

In [ ]:
# Enter your custom question
custom_question = "What is machine learning?"

response = requests.post(
    f"{api_url}/ask",
    json={"question": custom_question}
)
print(f"Question: {custom_question}")
print("\nResponse:")
print(json.dumps(response.json(), indent=2))

## Notes

- The system classifies questions into 5 domains: Math, Coding, Science, Reasoning, Agentic
- Difficulty levels: Easy, Medium, Hard
- The system automatically routes to the best chatbot based on classification
- If a chatbot fails, it falls back to OpenAI GPT-4o-mini
- All feedback is stored for future model improvements